# RELIANCE TikTok dataset

This notebook analyses the public RELIANCE `data.json` release. Its unit is an annotated sentence or paragraph, not a TikTok video row.

Sources: [official repository](https://github.com/realize-lab/RELIANCE), [project page](https://realize-lab.github.io/RELIANCE/), and [paper](https://arxiv.org/abs/2606.18285).

## 1. Load and verify the local official file

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'downloaded_datasets' / 'reliance'
JSON_PATH = DATA_DIR / 'data.json'
if not JSON_PATH.exists():
    raise FileNotFoundError(JSON_PATH)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

payload = json.loads(JSON_PATH.read_text())
data = pd.DataFrame(payload['data'])
display(pd.DataFrame([{
    'file': JSON_PATH.name,
    'bytes': JSON_PATH.stat().st_size,
    'KiB': JSON_PATH.stat().st_size / 1024,
    'sha256': sha256(JSON_PATH),
    'rows': len(data),
    'columns': len(data.columns),
}]).round({'KiB': 2}))
display(data.head(3))

,file,bytes,KiB,sha256,rows,columns
0,data.json,459676,448.90,65445a19cae23532d657795b2e271f51a329f1a6cac854...,409,6


,query,sentence,aggregated_acc,aggregated_harm,categories,stages
0,How does aspirin reduce preeclampsia risk?,Every 1000 people we treat with low dose aspir...,True,False,"[Labor and pain Management, Other, Supplement/...",[During Pregnancy]
1,How does aspirin reduce preeclampsia risk?,So preeclampsia is a condition in pregnancy th...,True,False,"[Labor and pain Management, Other, Supplement/...",[During Pregnancy]
2,How does aspirin reduce preeclampsia risk?,Low-Dose Aspirin Use During Pregnancy Table 1....,True,False,"[Labor and pain Management, Other, Supplement/...",[During Pregnancy]


## 2. Structure and comment availability

In [2]:
structure = pd.DataFrame([
    {'field': 'query', 'meaning': 'Health question used for collection'},
    {'field': 'sentence', 'meaning': 'Selected sentence or paragraph from a TikTok video'},
    {'field': 'aggregated_acc', 'meaning': 'Aggregated clinician accuracy label'},
    {'field': 'aggregated_harm', 'meaning': 'Aggregated clinician harmfulness label'},
    {'field': 'categories', 'meaning': 'One or more health-information categories'},
    {'field': 'stages', 'meaning': 'One or more reproductive-health stages'},
])
display(structure)

comment_columns = [column for column in data.columns if 'comment' in column.lower()]
id_columns = [column for column in data.columns if 'id' in column.lower() or 'url' in column.lower()]
print('Comment-related columns:', comment_columns)
print('Video ID or URL columns:', id_columns)
print('Mean/median comments per video can be calculated:', bool(comment_columns))

,field,meaning
0,query,Health question used for collection
1,sentence,Selected sentence or paragraph from a TikTok v...
2,aggregated_acc,Aggregated clinician accuracy label
3,aggregated_harm,Aggregated clinician harmfulness label
4,categories,One or more health-information categories
5,stages,One or more reproductive-health stages


Comment-related columns: []
Video ID or URL columns: []
Mean/median comments per video can be calculated: False


## 3. Check the published counts and released unit

In [3]:
count_checks = pd.DataFrame([
    {'measure': 'Annotated sentences/paragraphs', 'paper_or_README': 409, 'downloaded_or_verifiable': len(data)},
    {'measure': 'Clinician-reviewed queries', 'paper_or_README': 56, 'downloaded_or_verifiable': data['query'].nunique()},
    {'measure': 'Source TikTok videos', 'paper_or_README': 336, 'downloaded_or_verifiable': np.nan},
])
count_checks['note'] = [
    'Directly verifiable and identical',
    'The JSON contains 55 unique query strings; one of the 56 reported queries may have produced no released row',
    'Not verifiable because the JSON has no video ID or source-video field',
]
display(count_checks)

print('Duplicate sentence rows:', int(data['sentence'].duplicated().sum()))
print('Important: 409 rows must not be described as 409 unique videos.')

,measure,paper_or_README,downloaded_or_verifiable,note
0,Annotated sentences/paragraphs,409,409.00,Directly verifiable and identical
1,Clinician-reviewed queries,56,55.00,The JSON contains 55 unique query strings; one...
2,Source TikTok videos,336,NaN,Not verifiable because the JSON has no video I...


Duplicate sentence rows: 0
Important: 409 rows must not be described as 409 unique videos.


## 4. Accuracy and harmfulness labels

In [4]:
label_table = pd.crosstab(
    data['aggregated_acc'].map({True: 'Accurate', False: 'Inaccurate'}),
    data['aggregated_harm'].map({True: 'Harmful', False: 'Not harmful'}),
    margins=True,
)
display(label_table)

summary = pd.DataFrame([
    {'measure': 'Accurate text units', 'count': int(data['aggregated_acc'].sum()), 'percent': data['aggregated_acc'].mean() * 100},
    {'measure': 'Inaccurate text units', 'count': int((~data['aggregated_acc']).sum()), 'percent': (~data['aggregated_acc']).mean() * 100},
    {'measure': 'Harmful text units', 'count': int(data['aggregated_harm'].sum()), 'percent': data['aggregated_harm'].mean() * 100},
])
display(summary.round(2))

print('These are sentence/paragraph-level results from the released JSON. They are not video-level prevalence estimates.')

aggregated_harm,Harmful,Not harmful,All
aggregated_acc,,,
Accurate,0,337,337
Inaccurate,50,22,72
All,50,359,409


,measure,count,percent
0,Accurate text units,337,82.40
1,Inaccurate text units,72,17.60
2,Harmful text units,50,12.22


These are sentence/paragraph-level results from the released JSON. They are not video-level prevalence estimates.


## 5. Interesting point: number and length of text units per query

In [5]:
rows_per_query = data.groupby('query').size().rename('released_text_units')
display(rows_per_query.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]).to_frame())
display(rows_per_query.sort_values(ascending=False).head(10).to_frame())

data['word_count'] = data['sentence'].str.split().str.len()
display(data['word_count'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame())

,released_text_units
count,55.00
mean,7.44
std,3.48
min,1.00
25%,5.00
50%,7.00
75%,9.50
90%,12.00
95%,13.30
max,17.00


,released_text_units
query,
when does a woman start menstruation after giving birth?,17
when is rh immune globulin given during pregnancy?,15
Do certain foods make your baby smarter in the womb?,14
How can my partner help with labor pain?,13
when did they start testing for group b strep in pregnancy?,13
What foods should I avoid during pregnancy?,12
How does aspirin reduce preeclampsia risk?,12
How can I improve my fertility naturally?,12
Is it safe to travel by plane during pregnancy?,12


,word_count
count,409.00
mean,138.91
std,135.32
min,4.00
25%,52.00
50%,103.00
75%,178.00
90%,291.60
95%,379.60
99%,585.80


## 6. Categories and reproductive-health stages

In [6]:
category_counts = data.explode('categories')['categories'].value_counts().rename('label_occurrences').to_frame()
stage_counts = data.explode('stages')['stages'].value_counts().rename('label_occurrences').to_frame()
display(category_counts)
display(stage_counts)

print('Rows can have multiple categories and stages, so occurrence totals can exceed 409.')

,label_occurrences
categories,
Can I do <something>?,184
Physiology,173
Labor and pain Management,148
Symptoms,128
Supplement/Medication/Vaccines,114
Other,100
Diet/Exercise,98


,label_occurrences
stages,
During Pregnancy,290
Labor and Delivery,169
Trying to conceive,60
Postpartum and Infant care,44


Rows can have multiple categories and stages, so occurrence totals can exceed 409.


## 7. Missing values and list-label multiplicity

In [7]:
data['category_count'] = data['categories'].str.len()
data['stage_count'] = data['stages'].str.len()
display(pd.DataFrame({
    'categories_per_row': data['category_count'].value_counts().sort_index(),
    'stages_per_row': data['stage_count'].value_counts().sort_index(),
}).fillna(0).astype(int))

display(pd.DataFrame({
    'missing_count': data.isna().sum(),
    'missing_percent': data.isna().mean() * 100,
}).round(2))

,categories_per_row,stages_per_row
1,98,255
2,148,154
3,101,0
4,62,0


,missing_count,missing_percent
query,0,0.00
sentence,0,0.00
aggregated_acc,0,0.00
aggregated_harm,0,0.00
categories,0,0.00
stages,0,0.00
word_count,0,0.00
category_count,0,0.00
stage_count,0,0.00


# Findings

- The public JSON has **409 unique text units** and six fields. It is not a video-level metadata table.
- The 409-row count matches the paper and README, but the file has **55 unique query strings**, compared with 56 reviewed queries reported by the authors. A query with no released result is a plausible explanation, but the file does not prove the reason.
- The public file cannot verify the paper's 336-video count because it contains no video ID, URL, or source-video key.
- Accuracy and harm percentages in this notebook describe selected sentences or paragraphs. They should not be presented as the percentage of all sampled videos.
- The release contains no comments or comment-count field, so comments per video cannot be calculated.